![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 47 -- Case Study: SmartCargo (Starter)

The task of this morning's case study, **SmartCargo**, from the regional stage (OJIA) of the **Romanian AI Olympiad
2025**. Try it yourself: the statement, the data loaded and looked at, the score in code, and a first valid file that needs no model. Then follow the clues; the slides show how.

**Sources and credits.** The task, its statement, its data and its starter kit: Romanian AI Olympiad, OJIA 2025
([contest page](https://judge.nitro-ai.org/competitions/roai-2025/ojia)); the English translation of the statement and
the starter kit as published in [jaredliw/ioai-tsp-2025](https://github.com/jaredliw/ioai-tsp-2025/tree/main/onia-romania-2025/ojia/smartcargo)
(no licence stated), told here in our words with short quotes. The scoring function below is the organisers' own,
from that starter kit (comments translated). The data is downloaded at run time from S. Asandei's
[roai-solved](https://github.com/stefanasandei/roai-solved) (`national-contests/romania/regional/2025/smart-cargo`, MIT
licence), which also holds a 100-point solution. The rest of the code is the course's own, written for the Day 47
slides.

**How to run it.** Run every cell from the top; the data downloads in the Load cell. No GPU is needed.


## The task, in our words

**The story.** "SmartCargo Romania" delivers goods between Romanian cities and wants better estimates of how long a
trip takes. You are its new data scientist.

**The data.** `train_data.csv`: 10,000 past trips with their real delivery time `deliver_time` (minutes).
`test_data.csv`: 2,000 new trips without it. The columns: `ID`, `City A` (where the trip starts), `City B` (where it
ends), `Distance` (km), `Time of Day` (the minute of the day the trip starts, 0 to 1439), `Weather` (`Clear`, `Rain`,
`Snow`, `Fog`), `Traffic` (0 to 1000, higher is more crowded), `Road Quality` (1 to 1000, higher is better),
`Driver Experience` (1 to 40 years).

**Subtask 1 (20 points).** A client from Bârlad complains about delays in fog: how many test trips "depart from
Bârlad" in `Fog`? One whole number, right or wrong.

**Subtask 2 (80 points).** Estimate `deliver_time` for every test trip. Scored with MAE (mean absolute error), turned
into points by the function in the starter kit (below).

**The file.** One CSV with the columns `subtaskID`, `datapointID`, `answer`: subtask 1 is one row with `datapointID`
1; subtask 2 is one row per test trip, with its `ID` as `datapointID`.

**Lines worth a second look.** The statement writes Bârlad with an â; "depart from" means `City A`.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

## Load

In [ ]:
URL = ("https://raw.githubusercontent.com/stefanasandei/roai-solved/main/"
       "national-contests/romania/regional/2025/smart-cargo/")
train = pd.read_csv(URL + "train_data.csv")
test = pd.read_csv(URL + "test_data.csv")
print(train.shape, test.shape)     # (10000, 10) (2000, 9)
train.head()

## Look

In [ ]:
print(train["deliver_time"].describe())
print(train["Weather"].value_counts())
print(sorted(train["City A"].unique()))

## The score in code

In [ ]:
# The organisers' scoring function, from the starter kit (comments translated).
# It turns the MAE of subtask 2 into points: read it to know where the points stop.
def evaluate_t2_score(mae):
    # Interpolation such that at min_value you get score_min, at max_value you get score_max
    def lerp(value, min_value, max_value, score_min, score_max) -> int:
        slide = np.clip((value - min_value) / (max_value - min_value), 0.0, 1.0)
        score_res = score_min + (score_max - score_min) * slide
        return int(score_res)

    score = 0
    if mae >= 100:
        score = 0
    elif mae >= 20:
        score = lerp(mae, 100.0, 20.0, 0, 10)
    elif mae >= 10.0:
        score = lerp(mae, 20.0, 10.0, 10, 30)
    elif mae >= 8.0:
        score = lerp(mae, 10.0, 8.0, 30, 40)
    elif mae >= 5.0:
        score = lerp(mae, 8.0, 5.0, 40, 60)
    elif mae >= 4.5:
        score = lerp(mae, 5.0, 4.5, 60, 70)
    else:
        score = lerp(mae, 4.5, 4.3, 70, 80)

    return score, mae, score, mae


for mae in [30, 20.18, 10, 5, 4.3, 1.6]:
    print(mae, evaluate_t2_score(mae)[0])

## Two answers that need no model

In [ ]:
y = train["deliver_time"]
mae_mean = mean_absolute_error(y, [y.mean()] * len(y))     # the mean time for every trip
mae_dist = mean_absolute_error(y, train["Distance"])        # one minute per kilometre
print(round(mae_mean, 2), evaluate_t2_score(mae_mean)[0])
print(round(mae_dist, 2), evaluate_t2_score(mae_dist)[0])

## A first valid file

In [ ]:
# A first valid file that needs no model: subtask 1 still to find, time = distance
rows = [[1, 1, 0]]
for i in range(len(test)):
    rows.append([2, test["ID"].iloc[i], test["Distance"].iloc[i]])
submission = pd.DataFrame(rows, columns=["subtaskID", "datapointID", "answer"])
submission.to_csv("submission.csv", index=False)
print(len(submission))     # 2001
submission.head()

## Your turn

The file above is valid and scores 9 points of 80 for subtask 2, and nothing yet for subtask 1. Now follow the
clues of the slides, one at a time, each checked with code:

1. Subtask 1: is the city spelled in the data the way the statement spells it? Which column holds the departure?
2. Draw `deliver_time` against `Distance`. What is left once distance is taken out, and what does it move with?
3. Hold out validation trips, train a model, and score it with `evaluate_t2_score`. Where do the points stop?

Made By **Sattam Altwaim**